# ByteEmbed — FINAL retrieval study (BGE-M3 teacher, verified language set)

**The one notebook for all the finalized experiments** — run top-to-bottom: smoke → main 6-model
parallel run → both boundary arms → teacher baseline → AfriQA table.
The byte-vs-subword comparison is entirely within this run — both students share the teacher,
targets, data, recipe, and evaluation; only the tokenizer differs.

**Design (locked — see `RETRIEVAL_EXPERIMENT.md` for the full protocol):**
- **Languages (10):** te, bn, sw, yo, am, ha, rw (lower-resource; bn = Joshi class 3, the one stated
  relaxation) + en, zh, ar (anchors). ta/mr/so dropped. All 10 trained (~42k sentences each).
- **Students:** byt5 vs mt5 × {small, base, large} — 6 models, trained in parallel (`max_concurrent`)
- **Teacher:** **BGE-M3** (`BAAI/bge-m3`, retrieval-trained, 1024-d), targets cached once
- **Objective (retrieval-only):** pure InfoNCE (τ=0.05, queue 8192) — no alignment term, no
  relational term. AdamW lr 2e-4, **batch 64, 50k steps for every model (iso-step)**, `attn` pooling
- **Eval:** shallow = **Belebele only** (all 10); deep = **ONE benchmark per language** — MIRACL dev
  (te/bn/sw/yo/en/zh/ar) · Amharic-PR (am) · CIRAL Test A (ha, cross-lingual, flagged) ·
  **AfriQA (rw, cross-lingual REVERSE — rw question → English passage, flagged; the mirror of ha's
  standard)**. FLORES/Mr.TyDi/IndicQA/2AIRTC stay wired, off-default.
- **Boundary arms (step 5b, BOTH run):** `teacher` (real boundaries) + `random` (placebo) —
  the byte-only tokenization-mechanism probe.
- **AfriQA probe langs:** ha/sw/yo also get AfriQA numbers (reverse-axis viability) alongside rw's
  headline benchmark — all computed in the main eval.
- **Baseline:** **the teacher only** (BGE-M3 scored on the identical battery = the ceiling).
- **Caveat to report:** yo is not in XLM-R/CC-100 (the teacher's backbone) — weakest teacher signal;
  both students inherit it equally, so the comparison stays fair.

Everything is resumable; smoke first. First eval streams the CIRAL-ha corpus once (the big one-time
download) — pools cache to `checkpoints/` and every later model reuses them.

### 1. GPU check — confirm you're on an A100 (Runtime → Change runtime type → A100)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

### 2. Clone repo + install deps
Imports work from the repo root even if the editable install is skipped.

In [ ]:
import os
os.chdir('/content')
REPO = 'https://github.com/Aarushvinod/embedding-research.git'
if not os.path.isdir('/content/embedding-research'):
    !git clone -q $REPO
os.chdir('/content/embedding-research')
!git pull -q
!pip install -q -r requirements-cloud.txt
!pip install -q -e . || echo '(editable install skipped — running from repo root is fine)'
print('setup done | cwd', os.getcwd())

### 3. Teacher check + persist to Drive
BGE-M3 loads via sentence-transformers (already a dep) — no extra install, no fairseq2. Point
`PERSIST` at the **same** `byteembed_lowres` folder as the SONAR runs: the balanced-data cache is
reused; the BGE-M3 teacher targets get their own cache file (`teachertargets_bge-m3_*`), so nothing
collides. Skip the Drive block to run on ephemeral disk.

In [ ]:
from huggingface_hub import hf_hub_download
_ = hf_hub_download('BAAI/bge-m3', 'config.json')   # reachable? (weights download on first teacher load)
print('BGE-M3 reachable — teacher will be BAAI/bge-m3 (retrieval-trained, 1024-d)')

from google.colab import drive
drive.mount('/content/drive')
import os, shutil
PERSIST = '/content/drive/MyDrive/byteembed_lowres'   # SAME folder as the SONAR runs -> shared data cache
for d in ('results', 'checkpoints'):
    os.makedirs(f'{PERSIST}/{d}', exist_ok=True)
    if not os.path.islink(d):
        if os.path.isdir(d): shutil.rmtree(d)
        os.symlink(f'{PERSIST}/{d}', d)
print('persisting results/ and checkpoints/ to', PERSIST)

### 4. Smoke test (~5 min) — validate the pipeline with the NEW teacher
3 langs (am/sw/en), 2 tiny students, tiny eval. Confirms: BGE-M3 loads → targets precompute + cache →
train → full eval battery (incl. QA-retrieval) → save. **Check the log says 'BGE-M3 ... loaded' — not
SONAR, not LaBSE.**

In [ ]:
from byte_embed.run_lowresource import run
_ = run(smoke=True, teacher_name='bge-m3', pooling='attn', out='results/retrieval_bgem3_smoke.json')

### 5. Full retrieval study — 6 students, BGE-M3 targets
**50k steps × batch 64 for every model (iso-step), `attn` pooling for all.** The parallel runner
precomputes the BGE-M3 targets once, then trains several students at once. Resumable — finished
models skip; checkpoints carry a `_bge-m3` suffix (`byte-small_attn_bge-m3.pt`) so they never collide
with — or wrongly resume from — the SONAR-run checkpoints.

In [ ]:
from byte_embed.run_parallel import parallel
parallel(
    out='results/retrieval_bgem3.json',
    teacher_name='bge-m3',            # THE one change vs the SONAR study
    pooling='attn',                   # same pooling for byte AND subword (fair)
    steps=50000,                      # iso-step: 50k for every size (a CAP if patience is on)
    max_concurrent=3,                 # 80/96 GB card: 3-5
    patience=0,                       # loss-plateau early stop: 0 = off (exact iso-step).
    # min_delta=1e-3,                 #   e.g. patience=10 -> stop after 10x500 steps w/o improvement;
)                                     #   realized steps land in results as steps_run — report them.

# --- sequential fallback (one model at a time, live logs in-cell):
# from byte_embed.run_lowresource import run
# _ = run(out='results/retrieval_bgem3.json', teacher_name='bge-m3', pooling='attn', steps=50000)

### 5b. Boundary-injection arms — BOTH arms (byte-only mechanism probe)
Arm **B** (`teacher`): markers inserted where BGE-M3's tokenizer would split — segmentation info,
zero vocab table. Arm **C** (`random`): the placebo — same per-sentence marker count at random
character positions; separates "linguistic placement helps" from "any markers help". Teacher targets
stay clean; each arm trains AND evals with its own transform, in its own results file and
`_b-{arm}` checkpoint namespace. Runs on **byte-small** by default (each arm ≈ one small-model
training); widen `only=` to add sizes. Interpreting: B > C ≈ A → segmentation info helps;
B ≈ C > A → artifact, no credit to the tokenizer; B ≈ C ≈ A → byte needs nothing from segmentation.

In [ ]:
# BOTH boundary arms, back to back (resumable; each skips if already in its results file).
from byte_embed.run_lowresource import run
_ = run(out='results/retrieval_bgem3_bteacher.json', teacher_name='bge-m3', pooling='attn',
        steps=50000, boundary='teacher', only=['byte-small'])          # arm B — real boundaries
_ = run(out='results/retrieval_bgem3_brandom.json',  teacher_name='bge-m3', pooling='attn',
        steps=50000, boundary='random',  only=['byte-small'])          # arm C — random placebo

# Compare the three arms (A = the main run's byte-small) on the deep benchmarks:
import json
def _grab(path, name='byte-small'):
    try:
        r = json.load(open(path))['models'][name]
        mir = (r.get('miracl') or {}).get('ndcg@10_mean')
        qa = r.get('qa_retrieval') or {}
        am = (qa.get('amharicpr') or {}).get('ndcg@10_mean')
        ha = ((qa.get('ciral') or {}).get('per_lang', {}).get('ha') or {}).get('ndcg@10')
        bel = (r.get('means') or {}).get('belebele_ndcg@10')
        return dict(Belebele=bel, MIRACL=mir, AmharicPR=am, CIRAL_ha=ha)
    except Exception as e:
        return f'(pending: {type(e).__name__})'
for arm, path in [('A raw', 'results/retrieval_bgem3.json'),
                  ('B teacher-boundaries', 'results/retrieval_bgem3_bteacher.json'),
                  ('C random-boundaries', 'results/retrieval_bgem3_brandom.json')]:
    print(f'{arm:24}', _grab(path))

### 6. Teacher baseline + summary
The parallel runner trains students only. This cell scores **BGE-M3 itself** on the identical battery
(the per-benchmark ceiling the students chase — the only baseline we measure) and prints the full
table: Belebele / FLORES / MIRACL + the deep-QA block (Amharic-PR am · CIRAL ha).

In [ ]:
from byte_embed.run_lowresource import run
_ = run(out='results/retrieval_bgem3.json', teacher_name='bge-m3', pooling='attn', steps=50000)

### 6b. AfriQA table (rw's deep benchmark + the ha/sw/yo reverse-axis probe)
AfriQA now runs **in the main eval** (default battery): **rw is its headline** — Kinyarwanda's deep
benchmark (347 native questions → English gold passages + 20k English distractors; flagged
cross-lingual-REVERSE, the mirror of ha's CIRAL standard) — with ha/sw/yo riding along as the
reverse-axis probe. This cell backfills any models evaluated before AfriQA joined the battery
(additive — existing metrics kept) and prints the per-language table.

In [ ]:
# AfriQA backfill (no-op if the main run already computed it) + per-language table.
from byte_embed.reeval import reeval
reeval('results/retrieval_bgem3.json', pooling='attn', qa_only=True, benchmarks=('afriqa',))

# African question -> English passage; rw is the headline (its deep benchmark), ha/sw/yo = probe:
import json
res = json.load(open('results/retrieval_bgem3.json'))
print(f"{'model':16}" + "".join(f"{l:>10}" for l in ('rw', 'ha', 'sw', 'yo')))
for name, r in res['models'].items():
    per = ((r.get('qa_retrieval') or {}).get('afriqa') or {}).get('per_lang') or {}
    row = "".join(f"{(per.get(l) or {}).get('ndcg@10', float('nan')):>10.3f}" if per.get(l) else f"{'-':>10}"
                  for l in ('rw', 'ha', 'sw', 'yo'))
    print(f"{name:16}{row}")

### 7. Download results
Already on Drive if you ran the persist cell; otherwise grab the JSON here.

In [ ]:
from google.colab import files
files.download('results/retrieval_bgem3.json')
for f in ('results/retrieval_bgem3_bteacher.json', 'results/retrieval_bgem3_brandom.json'):
    try: files.download(f)
    except Exception as e: print('skip', f, '-', e)